In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path('/cta/users/guneyn23')
OUT = BASE / 'relative_repair/window_median_relative_all_regions'
OUT.mkdir(parents=True, exist_ok=True)

REGIONS = ['ATAC', 'H3K9me3', 'H3K27me3']
DAMAGES = ['CPD', '64']
TIMES = ['15m', '30m', '1h', '4h', '8h']
COLORS = {'15m': '#0072B2', '30m': '#E69F00', '1h': '#009E73', '4h': '#D55E00', '8h': '#CC79A7'}

BASELINES = {
    ('ATAC', 'CPD', 'real'): BASE / 'rpkm_1m/CPD_rpkm/atac_rpkm/real_damage_rpkm.bed',
    ('ATAC', 'CPD', 'sim'): BASE / 'rpkm_1m/CPD_rpkm/atac_rpkm/sim_damage_rpkm.bed',
    ('ATAC', '64', 'real'): BASE / 'rpkm_1m/64_rpkm/ATAC_real_64_rpkm.bed',
    ('ATAC', '64', 'sim'): BASE / 'rpkm_1m/64_rpkm/ATAC_simulated_64_rpkm.bed',
    ('H3K9me3', 'CPD', 'real'): BASE / 'rpkm_1m/CPD_rpkm/close_rpkm/H3k9me3_real_rpkm.bed',
    ('H3K9me3', 'CPD', 'sim'): BASE / 'rpkm_1m/CPD_rpkm/close_rpkm/H3k9me3_sim_rpkm.bed',
    ('H3K9me3', '64', 'real'): BASE / 'rpkm_1m/64_rpkm/H3K9me3_real_64_rpkm.bed',
    ('H3K9me3', '64', 'sim'): BASE / 'rpkm_1m/64_rpkm/H3K9me3_simulated_64_rpkm.bed',
    ('H3K27me3', 'CPD', 'real'): BASE / 'rpkm_1m/CPD_rpkm/close_rpkm/H3k27me3_real_rpkm.bed',
    ('H3K27me3', 'CPD', 'sim'): BASE / 'rpkm_1m/CPD_rpkm/close_rpkm/H3k27me3_sim_rpkm.bed',
    ('H3K27me3', '64', 'real'): BASE / 'rpkm_1m/64_rpkm/H3K27me3_real_64_rpkm.bed',
    ('H3K27me3', '64', 'sim'): BASE / 'rpkm_1m/64_rpkm/H3K27me3_simulated_64_rpkm.bed',
}

STEMS = {
    ('CPD', '15m'): 'R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS',
    ('CPD', '30m'): 'R3Hela_30mCPD_GGCTAC_S8_hg38_primary_assembly_DS',
    ('CPD', '1h'): 'R3Hela_1hCPD_CTTGTA_S4_hg38_primary_assembly_DS',
    ('CPD', '4h'): 'R3Hela_4hCPD_AGTCAA_S10_hg38_primary_assembly_DS',
    ('CPD', '8h'): 'R3Hela_8hCPD_AGTTCC_S12_hg38_primary_assembly_DS',
    ('64', '15m'): 'R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS',
    ('64', '30m'): 'R3Hela_30m64_TGACCA_S7_hg38_primary_assembly_DS',
    ('64', '1h'): 'R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS',
    ('64', '4h'): 'R3Hela_4h64_GCCAAT_S9_hg38_primary_assembly_DS',
    ('64', '8h'): 'R3Hela_8h64_CAGATC_S11_hg38_primary_assembly_DS',
}

def timepoint_path(region, damage, time, simulated=False):
    stem = STEMS[(damage, time)] + ('_sim' if simulated else '')
    if region == 'ATAC':
        folder = BASE / 'peak_center_20kb/ATAC_rpkm/noUV_atac_damage_all'
        return folder / f'{stem}_noUV_ATAC_400windows_rpkm.bed'
    folder = BASE / 'peak_center_20kb/close_rpkm' / region
    return folder / f'{stem}_{region}_400windows_rpkm.bed'

def medians_by_window(path):
    data = pd.read_csv(path, sep='\t', header=None, usecols=[3, 5], names=['peak', 'rpkm'])
    data['window'] = data['peak'].str.rsplit('_', n=1).str[-1].astype(int)
    return data.groupby('window')['rpkm'].median()

def relative_repair(baseline, timepoint):
    return ((baseline - timepoint) / baseline).where(baseline != 0)

In [2]:
results = []

for region in REGIONS:
    for damage in DAMAGES:
        baseline_real = medians_by_window(BASELINES[(region, damage, 'real')])
        baseline_sim = medians_by_window(BASELINES[(region, damage, 'sim')])

        for time in TIMES:
            time_real = medians_by_window(timepoint_path(region, damage, time))
            time_sim = medians_by_window(timepoint_path(region, damage, time, simulated=True))

            result = pd.concat({
                'median_baseline_real': baseline_real,
                'median_time_real': time_real,
                'median_baseline_sim': baseline_sim,
                'median_time_sim': time_sim,
            }, axis=1).reset_index()

            result.insert(0, 'time', time)
            result.insert(0, 'damage', damage)
            result.insert(0, 'region', region)
            result['relative_real'] = relative_repair(result['median_baseline_real'], result['median_time_real'])
            result['relative_sim'] = relative_repair(result['median_baseline_sim'], result['median_time_sim'])
            result['relative_real_div_sim'] = (result['relative_real'] / result['relative_sim']).replace([np.inf, -np.inf], np.nan)

            result.to_csv(OUT / f'{region}_{damage}_{time}_median_relative.tsv', sep='\t', index=False)
            results.append(result)

summary = pd.concat(results, ignore_index=True)
summary.to_csv(OUT / 'all_chromatin_median_relative_results.tsv', sep='\t', index=False)
summary.head()

KeyboardInterrupt: 

In [ ]:
PLOTS = {
    'relative_real_div_sim': ('Real / simulated relative repair', 'real_div_sim'),
    'relative_real': ('Real relative repair', 'real'),
    'relative_sim': ('Simulated relative repair', 'sim'),
}

for region in REGIONS:
    for damage in DAMAGES:
        selected = summary.query('region == @region and damage == @damage')

        for column, (label, suffix) in PLOTS.items():
            fig, ax = plt.subplots(figsize=(10, 5))
            for time in TIMES:
                values = selected.query('time == @time').sort_values('window')
                
                x = -10 + (values['window'] - 0.5) * (20 / 400)
                ax.plot(x, values[column], color=COLORS[time], label=time)

            ax.axhline(0, color='0.75', linewidth=0.8)
            ax.axvline(0, color='0.5', linestyle='--')
            ax.set(xlim=(-10, 10), xlabel=f'Distance from {region} region center (kb)',
                   ylabel=label, title=f'{region} {damage}: {label}')
            ax.legend(title='Time')
            fig.tight_layout()
            fig.savefig(OUT / f'{region}_{damage}_{suffix}.png', dpi=300, bbox_inches='tight')
            plt.show()
            plt.close(fig)